In [109]:
## Import
import numpy as np
from scipy.fftpack import fft2, ifft2
from scipy.integrate import solve_ivp
#import matplotlib.pyplot as plt
from scipy.linalg import kron

In [110]:
# Initial Conditions
beta = 1
D1 = 0.1
D2 = 0.1
m = 1 
tspan = np.linspace(0, 4, 9)  

In [111]:
## Get A1
# With periodic boundaries

Lx, Ly = 20, 20
nx, ny = 64, 64
N = nx * ny

x = np.linspace(-Lx/2, Lx/2, nx + 1); x = x[:nx]
y = np.linspace(-Ly/2, Ly/2, ny + 1); y = y[:ny]
X, Y = np.meshgrid(x, y)

# Define U and V
U = np.tanh(np.sqrt(X**2 + Y**2)) * np.cos(m * np.angle(X + 1j * Y) - np.sqrt(X**2 + Y**2))
V = np.tanh(np.sqrt(X**2 + Y**2)) * np.sin(m * np.angle(X + 1j * Y) - np.sqrt(X**2 + Y**2))
Ut = fft2(U)
Vt = fft2(V)
U0 = Ut.reshape(N)
V0 = Vt.reshape(N)
init = np.hstack((U0, V0))

# Define spectral k values
kx = ((2 * np.pi) / Lx) * np.concatenate((np.arange(0, nx/2), np.arange(-nx/2, 0)))
ky = ((2 * np.pi) / Ly) * np.concatenate((np.arange(0, ny/2), np.arange(-ny/2, 0)))
kx[0] = 1e-6; ky[0] = 1e-6
KX, KY = np.meshgrid(kx, ky)
K = KX ** 2 + KY ** 2

# Using FFT
def fft_rhs(t, init2):
    init_U = init2[0:N] 
    init_V = init2[N:]
    init_U = init_U.reshape((nx, ny))
    init_V = init_V.reshape((nx, ny))

    U = ifft2(init_U)
    V = ifft2(init_V)
    
    lambdaA = 1 - (U**2 + V**2)
    omegaA = -beta * (U**2 + V**2)

    U_t = fft2(lambdaA * U - omegaA * V) - D1 * K * init_U
    V_t = fft2(omegaA * U + lambdaA * V) - D2 * K * init_V
    U_t = U_t.reshape(N)
    V_t = V_t.reshape(N)
    rhs = np.hstack((U_t, V_t))

    return rhs


sol = solve_ivp(fft_rhs, [tspan[0], tspan[-1]], init, t_eval=tspan, method='RK45')
sol = sol.y

A1 = sol; print(A1)
A1.shape

[[ 24.94003847+0.00000000e+00j  12.73268299-3.34833145e-16j
   -1.38095598-9.96305604e-15j ... -64.02389647-3.56394442e-14j
  -67.76356741-2.67048015e-14j -61.18058974-3.34135546e-15j]
 [-18.55666362-5.81663109e+01j -42.51586944-4.69129224e+01j
  -60.80795253-2.57480390e+01j ... -26.39439597+1.13082890e+02j
    6.86544434+1.23000456e+02j  41.4436393 +1.10055312e+02j]
 [-16.04755868+3.28279829e+01j -22.03971648-4.57977740e+01j
  -23.23089505-1.04141716e+02j ... -25.03391682-9.26527314e+01j
  -29.2936105 -4.09594873e+01j -31.3712619 +1.56986891e+01j]
 ...
 [ 24.73021466-5.66774723e+02j  34.94179045-3.31372917e+02j
   38.82924248-4.97842318e+01j ...   4.99619196+6.02396295e+02j
   -9.93322885+4.90736906e+02j -25.6299042 +2.81792021e+02j]
 [ 25.33720124-3.61633792e+02j  43.00958768-4.53711746e+02j
   51.93221654-4.47841562e+02j ... -30.76392977+2.66442187e+02j
  -58.45411318+4.29165358e+02j -74.0191717 +5.05315322e+02j]
 [ -6.4753501 +3.96245454e+01j  15.86720969-5.83358549e+01j
   37.7389

(8192, 9)

In [112]:
## Get A2
# with no-flux boundaries
N = 30
m = 1

# Using Chebychev 
def cheb(N):
	if N==0: 
		D = 0.; x = 1.
	else:
		n = np.arange(0,N+1)
		x = np.cos(np.pi*n/N).reshape(N+1,1) 
		c = (np.hstack(( [2.], np.ones(N-1), [2.]))*(-1)**n).reshape(N+1,1)
		X = np.tile(x,(1,N+1))
		dX = X - X.T
		D = np.dot(c,1./c.T)/(dX+np.eye(N+1))
		D -= np.diag(np.sum(D.T,axis=0))
	return D, x.reshape(N+1)


# Chebyshev differentiation matrix and grid
D, x = cheb(N)
D[N, :] = 0
D[0, :] = 0
D_xx = np.dot(D, D) / (10 ** 2)
y = x
N2 = (N + 1) * (N + 1)
I = np.eye(len(D_xx))
L = kron(I, D_xx) + kron(D_xx, I) # 2D laplacian 
X, Y = np.meshgrid(x, y)
X = X * 10; Y = Y * 10

# Define U and V
U = np.tanh(np.sqrt(X**2 + Y**2)) * np.cos(m * np.angle(X + 1j * Y) - np.sqrt(X**2 + Y**2))
V = np.tanh(np.sqrt(X**2 + Y**2)) * np.sin(m * np.angle(X + 1j * Y) - np.sqrt(X**2 + Y**2))
init_A2 = np.hstack((U.reshape(N2), V.reshape(N2)))

def RD_2D(t, init2):
	U = init2[0: N2]
	V = init2[N2:]

	lambdaA = 1 - (U ** 2 + V ** 2)
	omegaA = -beta * (U**2 + V**2)

	rhs_u = D1 * np.dot(L, U) + lambdaA * U - omegaA * V
	rhs_v = D2 * np.dot(L, V) + omegaA * U + lambdaA * V

	rhs = np.hstack([rhs_u, rhs_v])
	return rhs


sol2 = solve_ivp(RD_2D, [tspan[0], tspan[-1]], init_A2, t_eval=tspan, method='RK45')
sol2 = sol2.y

A2 = sol2; print(A2)
A2.shape



[[ 0.70358468  0.27678435 -0.21775865 ... -0.79689015 -0.40972859
   0.07776933]
 [ 0.73241275  0.47188952  0.07344742 ... -0.96577657 -0.78500366
  -0.4261521 ]
 [ 0.81058026  0.37605887 -0.11123233 ... -0.84008598 -0.49565779
  -0.03085913]
 ...
 [ 0.58562756  0.91352592  0.97914313 ... -0.50294695 -0.84298442
  -0.97634716]
 [ 0.6808609   0.87018536  0.97997159 ... -0.16453512 -0.5878894
  -0.88455009]
 [ 0.71061143  0.96093661  0.97601586 ... -0.60413504 -0.91222169
  -0.99697897]]


(1922, 9)